## Cell 1: setup

In [1]:
import sys
from pathlib import Path
import numpy as np, pandas as pd, rasterio
from PIL import Image

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "backend")); sys.path.insert(0, str(ROOT / "eval"))
from pipeline import predict_relative_depth
from pipeline.segmentation import remap_dfc_labels
import metrics as M

RAW = ROOT / "data" / "raw" / "dfc2019"
CACHE = ROOT / "backend" / "outputs" / "cache"; CACHE.mkdir(parents=True, exist_ok=True)
N_TILES = 30
CLASSES = {0: "ground", 1: "vegetation", 2: "building", 3: "water"}

## Cell 2: load tiles (depth cached to disk, so reruns are fast)

In [2]:
def find_tiles():
    out = []
    for rgb in sorted(RAW.rglob("*_RGB.tif")):
        base = rgb.name[:-len("_RGB.tif")]
        agl = next(RAW.rglob(f"{base}_AGL.tif"), None)
        cls = next(RAW.rglob(f"{base}_CLS.tif"), None)
        if agl and cls:
            out.append((base, rgb, agl, cls))
    return out

def read_rgb(path):
    with rasterio.open(path) as src:
        arr = src.read([1, 2, 3]).transpose(1, 2, 0).astype(np.float32)
    if arr.max() > 255:
        lo, hi = np.percentile(arr, (2, 98))
        arr = np.clip((arr - lo) / (hi - lo + 1e-6), 0, 1) * 255
    return Image.fromarray(arr.astype(np.uint8))

def read_band(path):
    with rasterio.open(path) as src:
        return src.read(1)

tiles = find_tiles()
idx = np.linspace(0, len(tiles) - 1, N_TILES).astype(int)
data = {}
for name, rgb_p, agl_p, cls_p in [tiles[i] for i in idx]:
    f = CACHE / f"{name}.npz"
    if f.exists():
        z = np.load(f); data[name] = (z["depth"], z["agl"], z["labels"]); continue
    depth = predict_relative_depth(read_rgb(rgb_p))
    agl = read_band(agl_p).astype(np.float32)
    labels = remap_dfc_labels(read_band(cls_p))
    np.savez_compressed(f, depth=depth, agl=agl, labels=labels)
    data[name] = (depth, agl, labels)

print(len(data), "tiles ready")
g = np.concatenate([agl[lab == 0] for _, agl, lab in data.values()])
print("Mean AGL on ground pixels (should be ~0):", round(float(np.nanmean(g)), 2))

/home/tanmay/Projects/depthWizard/depthWizard/.venv/lib/python3.14/site-packages/rasterio/__init__.py:367: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, thread_safe=thread_safe, **kwargs)


Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

30 tiles ready
Mean AGL on ground pixels (should be ~0): 0.47


## Cell 3: calibration methods

In [3]:
def fit_line(x, y, robust=False, iters=10):
    a, b = np.polyfit(x, y, 1)
    if robust:  # Huber IRLS
        for _ in range(iters):
            r = y - (a * x + b)
            s = 1.4826 * np.median(np.abs(r - np.median(r))) + 1e-6
            k = 1.345 * s
            w = np.where(np.abs(r) <= k, 1.0, k / np.maximum(np.abs(r), 1e-9))
            A = np.stack([x, np.ones_like(x)], 1) * np.sqrt(w)[:, None]
            a, b = np.linalg.lstsq(A, y * np.sqrt(w), rcond=None)[0]
    return a, b

def calibrate(method, d, lab, ref_d, ref_y, ref_l, min_pts=15, shrink_k=30):
    robust = "robust" in method
    ga, gb = fit_line(ref_d, ref_y, robust)
    out = ga * d + gb
    if method.startswith("global"):
        return out
    for c in CLASSES:
        m = ref_l == c
        n = int(m.sum())
        if n < min_pts or np.ptp(ref_d[m]) < 1e-6:
            continue                      # too few points: keep the global fit
        a, b = fit_line(ref_d[m], ref_y[m], robust)
        if "shrunk" in method:            # pull small-sample class fits toward global
            w = n / (n + shrink_k)
            a, b = w * a + (1 - w) * ga, w * b + (1 - w) * gb
        out[lab == c] = a * d[lab == c] + b
    return out

## Cell 4: run the experiment

In [4]:
METHODS = ["global", "global_robust", "perclass", "perclass_robust", "perclass_robust_shrunk"]
N_REF, SEEDS = 256, range(5)
rows = []

def record(name, seed, method, pred, agl, lab, held):
    m = M.compute_metrics(pred, agl, mask=held)
    pc = M.per_class_metrics(pred, agl, lab, CLASSES)
    rows.append({"tile": name, "seed": seed, "method": method,
                 "rmse": m["rmse"], "mae": m["mae"], "bias": m["bias"],
                 **{f"rmse_{k}": v["rmse"] for k, v in pc.items()}})

for name, (d, agl, lab) in data.items():
    valid = np.isfinite(agl) & (lab != 255)
    vi = np.flatnonzero(valid.ravel())

    for seed in SEEDS:                                   # sparse reference
        pick = np.random.default_rng(seed).choice(vi, N_REF, replace=False)
        r, c = np.unravel_index(pick, d.shape)
        held = valid.copy(); held[r, c] = False
        for method in METHODS:
            pred = calibrate(method, d, lab, d[r, c], agl[r, c], lab[r, c])
            record(name, seed, method, pred, agl, lab, held)

    pick = np.random.default_rng(99).choice(vi, 50_000, replace=False)   # oracle ceiling
    r, c = np.unravel_index(pick, d.shape)
    held = valid.copy(); held[r, c] = False
    for method, label in [("global_robust", "oracle_global"), ("perclass_robust", "oracle_perclass")]:
        pred = calibrate(method, d, lab, d[r, c], agl[r, c], lab[r, c], min_pts=15)
        record(name, 0, label, pred, agl, lab, held)

df = pd.DataFrame(rows)
print("done:", len(df), "rows")

done: 810 rows


## Cell 5: results

In [5]:
order = METHODS + ["oracle_global", "oracle_perclass"]
per_tile = df.groupby(["method", "tile"]).mean(numeric_only=True).reset_index()

summary = per_tile.groupby("method").agg(
    rmse_mean=("rmse", "mean"), rmse_median=("rmse", "median"), mae=("mae", "mean"),
    ground=("rmse_ground", "mean"), vegetation=("rmse_vegetation", "mean"),
    building=("rmse_building", "mean"), water=("rmse_water", "mean")).round(2)
display(summary.loc[order])

piv = per_tile.pivot(index="tile", columns="method", values="rmse")
print("\nShare of tiles where method beats plain global:")
for m in order[1:]:
    print(f"  {m:24s} {(piv[m] < piv['global']).mean():.2f}")

print("\nWorst 5 tiles (oracle_global):")
display(per_tile[per_tile.method == "oracle_global"].nlargest(5, "rmse")[["tile", "rmse"]])

,rmse_mean,rmse_median,mae,ground,vegetation,building,water
method,,,,,,,
global,3.70,3.44,2.51,2.47,5.37,4.37,1.36
global_robust,3.84,3.66,2.30,2.12,5.88,5.07,1.32
perclass,2.94,2.72,1.63,1.50,4.35,3.79,1.14
perclass_robust,3.01,2.80,1.46,1.60,4.40,4.06,1.15
perclass_robust_shrunk,3.14,2.98,1.58,1.55,4.77,4.34,1.22
oracle_global,3.82,3.65,2.30,2.12,5.85,5.02,1.27
oracle_perclass,2.93,2.73,1.41,1.60,3.96,3.53,0.76



Share of tiles where method beats plain global:
  global_robust            0.03
  perclass                 1.00
  perclass_robust          0.97
  perclass_robust_shrunk   0.93
  oracle_global            0.10
  oracle_perclass          1.00

Worst 5 tiles (oracle_global):


,tile,rmse
64,JAX_165_014,8.429251
67,JAX_251_004,5.732058
65,JAX_175_015,5.731519
79,OMA_248_025,5.234493
66,JAX_207_005,4.964977


## Cell 6: nonlinear method (quadratic per class, with safeguards)

In [6]:
def calibrate_quad(d, lab, ref_d, ref_y, ref_l, min_pts=40, min_lin=15):
    """Per-class quadratic. Falls back to linear for small classes, and wherever the
    curve would not be monotonically increasing across the class's full depth range."""
    ga, gb = fit_line(ref_d, ref_y)
    out = ga * d + gb
    for c in CLASSES:
        m = ref_l == c
        n = int(m.sum())
        sel = lab == c
        if n < min_lin or np.ptp(ref_d[m]) < 1e-6 or not sel.any():
            continue
        a, b = fit_line(ref_d[m], ref_y[m])
        pred = a * d[sel] + b
        if n >= min_pts:
            q2, q1, q0 = np.polyfit(ref_d[m], ref_y[m], 2)
            lo, hi = d[sel].min(), d[sel].max()
            if (2 * q2 * lo + q1) > 0 and (2 * q2 * hi + q1) > 0:
                pred = q2 * d[sel] ** 2 + q1 * d[sel] + q0
        out[sel] = pred
    return out

In [7]:
for name, (d, agl, lab) in data.items():
    valid = np.isfinite(agl) & (lab != 255)
    vi = np.flatnonzero(valid.ravel())

    for seed in SEEDS:
        pick = np.random.default_rng(seed).choice(vi, N_REF, replace=False)
        r, c = np.unravel_index(pick, d.shape)
        held = valid.copy(); held[r, c] = False
        pred = calibrate_quad(d, lab, d[r, c], agl[r, c], lab[r, c])
        record(name, seed, "perclass_quad", pred, agl, lab, held)

    pick = np.random.default_rng(99).choice(vi, 50_000, replace=False)
    r, c = np.unravel_index(pick, d.shape)
    held = valid.copy(); held[r, c] = False
    record(name, 0, "oracle_lin", calibrate("perclass", d, lab, d[r, c], agl[r, c], lab[r, c]), agl, lab, held)
    record(name, 0, "oracle_quad", calibrate_quad(d, lab, d[r, c], agl[r, c], lab[r, c]), agl, lab, held)

df = pd.DataFrame(rows)
order2 = ["global", "perclass", "perclass_quad", "oracle_lin", "oracle_quad"]
per_tile = df.groupby(["method", "tile"]).mean(numeric_only=True).reset_index()

summary = per_tile.groupby("method").agg(
    rmse_mean=("rmse", "mean"), rmse_median=("rmse", "median"), mae=("mae", "mean"),
    ground=("rmse_ground", "mean"), vegetation=("rmse_vegetation", "mean"),
    building=("rmse_building", "mean"), water=("rmse_water", "mean")).round(2)
display(summary.loc[order2])

piv = per_tile.pivot(index="tile", columns="method", values="rmse")
print("Share of tiles where quad beats linear per-class:",
      round((piv["perclass_quad"] < piv["perclass"]).mean(), 2))

,rmse_mean,rmse_median,mae,ground,vegetation,building,water
method,,,,,,,
global,3.70,3.44,2.51,2.47,5.37,4.37,1.36
perclass,2.94,2.72,1.63,1.50,4.35,3.79,1.14
perclass_quad,2.94,2.72,1.63,1.50,4.35,3.78,1.14
oracle_lin,2.86,2.65,1.59,1.48,3.95,3.50,0.71
oracle_quad,2.85,2.65,1.59,1.48,3.95,3.50,0.71


Share of tiles where quad beats linear per-class: 0.27
